# CS383: Data Science and Machine Learning
## Lecture 11 — Cross-Validation, Pipelines, Tuning, and Regularization

*Dr. Thitima Srivatanakul*

### Guiding question
**Your model scored well on one train/test split. Would it score just as well on a different random
split? And how do you find the best settings for a model without ever peeking at the test set?**

### Learning objectives
By the end of this lecture, you should be able to:

- explain why a single train/test split gives a noisy estimate of performance, and how k-fold
  cross-validation averages over several splits for a more honest one
- use `Pipeline` to bundle preprocessing and a model together, and explain why that matters even more
  once you start cross-validating or searching over hyperparameters
- use `GridSearchCV` to search over hyperparameter values (like `k` in k-NN or `alpha` in Ridge/Lasso)
  using only the training data, never touching the test set until the very end
- explain what Ridge and Lasso regularization do to a linear model's coefficients, and why that can help
  a model generalize
- build a tuned `Pipeline` on real NYC 311 data and evaluate it fairly, exactly once, on a held-out test
  set

### Where this fits
Lecture 7 introduced the train/test split and a plain linear regression baseline. Lectures 8-10 each
built a classifier and evaluated it once, on one split. Today formalizes a concern that's been lurking
since Lecture 7: one split is one roll of the dice. Cross-validation, `Pipeline`, and `GridSearchCV` are
the practical toolkit for tuning and validating a model honestly, and Ridge/Lasso give you your first
tool for actively fighting overfitting in a linear model, rather than just diagnosing it after the fact.

---

### Before we open the notebook: an unplugged warm-up

No coding for this part. You'll split a small dataset by hand several different ways to see how much a
single split's score can wobble, walk through k-fold cross-validation one fold at a time, and watch what
happens to a set of coefficients as a regularization "dial" is turned up.

**[Open the "One Split Isn't Enough" Activity](https://thitimas.github.io/cs383-fa26-materials-public/lect11/pipelines_tuning_unplugged_activity.html)**

Takes about 15-20 minutes. Come back here once you've been through all the rounds.

---

## Part 1 — One Split Isn't Enough

Every model so far has been scored on exactly one train/test split. Before doing anything else, let's see
how much that one number can wobble just from *which* rows happened to land in the test set.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, classification_report
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

# A small synthetic dataset -- just noisy enough, and just small enough (n=40), that which rows end up
# in the test set will visibly matter.
rng = np.random.default_rng(383)
n = 40
toy_X = rng.normal(0, 1, size=(n, 1))
toy_y = 3 * toy_X[:, 0] + rng.normal(0, 2.5, size=n)

print(f"{n} rows, 1 feature, true relationship: y = 3x + noise")

In [ ]:
for rs in range(5):
    X_tr, X_te, y_tr, y_te = train_test_split(toy_X, toy_y, test_size=0.3, random_state=rs)
    m = LinearRegression().fit(X_tr, y_tr)
    r2 = __________(y_te, m.predict(X_te))
    print(f"random_state={rs}: test R² = {r2:.3f}")

Same data, same model, same code -- only `random_state` changed which 12 rows landed in the test set.
The R² swings around a lot, sometimes a little, sometimes a lot more, purely because of that luck of the
draw. Nothing about the model changed between runs; only the sample of points used to judge it did. With
a test set this small, one split is closer to a coin flip than a verdict.

That's the problem cross-validation exists to solve.

---

## Part 2 — K-Fold Cross-Validation

Instead of judging a model on one lucky-or-unlucky split, **k-fold cross-validation** runs the split
several times and averages the result:

1. Split the data into *k* equal-sized folds (5 is a common choice).
2. For each fold in turn: train on the other *k*-1 folds, test on that one fold.
3. You now have *k* scores instead of 1. Average them.

Every row gets used for testing exactly once, and for training *k*-1 times -- so the final number reflects
the whole dataset, not just whichever 30% happened to land in the test set that one time.

In [ ]:
cv_model = LinearRegression()
fold_scores = __________(cv_model, toy_X, toy_y, cv=5, scoring="r2")

print("Per-fold R²:", np.round(fold_scores, 3))
print(f"Mean R²: {fold_scores.mean():.3f}  (+/- {fold_scores.std():.3f})")

One call to `cross_val_score` did all 5 splits, fit and scored 5 models, and handed back 5 numbers.
Notice the mean CV R² can land *lower* than every single split score from Part 1 -- that's not a
contradiction, it's the point. Part 1's `random_state=2` split scored 0.718 only because that particular
12-row test set happened to avoid the data's harder points. Cross-validation forces every row to take a
turn being tested, including the ones that make some fold genuinely hard (one fold above may even come out
negative) -- so its mean is a less flattering but more honest picture than a lucky single split.

The spread across folds matters too: a large standard deviation (as you'll likely see here, given only 40
rows split five ways) is itself useful information a single train/test split could never give you at all
-- it's telling you this estimate isn't very precise yet, which is a legitimate signal to want more data,
not a flaw in the method.

This is why, from here on, "how good is this model?" should usually be answered with a cross-validated
mean *and* its spread, not a single split's score.

---

## Part 3 — Pipelines Prevent Leakage During Cross-Validation

Lecture 7 taught the leakage rule for a single train/test split: fit any preprocessing (scalers,
encoders) on the training data only, then just *apply* it to the test data. Cross-validation introduces a
subtler version of the same trap. There isn't one train/test split anymore -- there are *k* of them, one
per fold, generated internally as `cross_val_score` runs. If you fit a scaler on the **entire** dataset
before handing it to `cross_val_score`, then every fold's "held-out" portion already influenced that
scaler's mean and standard deviation, before that fold was ever tested on.

The fix is the same tool for a different reason: put preprocessing **inside** a `Pipeline`. Handed to
`cross_val_score`, a `Pipeline` refits its preprocessing step fresh on each fold's training portion, and
only *applies* (never refits) it to that fold's held-out portion -- automatically, for every fold.

In [ ]:
# LEAKY -- for illustration only. Scaling the *entire* dataset before cross-validation means every
# fold's "held-out" portion already influenced the scaler's mean/std before being tested on.
leaky_scaler = StandardScaler()
toy_X_scaled_leaky = leaky_scaler.__________(toy_X)

leaky_scores = cross_val_score(LinearRegression(), toy_X_scaled_leaky, toy_y, cv=5, scoring="r2")
print(f"Leaky CV mean R²: {leaky_scores.mean():.3f}")

In [ ]:
# CORRECT -- scaling lives inside the Pipeline, so cross_val_score refits it fresh on each fold's
# training portion, and only applies (never refits) it to that fold's held-out portion.
safe_pipeline = __________([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression()),
])
safe_scores = cross_val_score(safe_pipeline, toy_X, toy_y, cv=5, scoring="r2")
print(f"Pipeline CV mean R²: {safe_scores.mean():.3f}")

These two numbers likely come out *identical*. That's not a mistake -- for plain linear regression,
rescaling one numeric feature by a constant factor doesn't change R² at all, so this particular
leak happens to be invisible for this exact model-and-preprocessing combination. That doesn't mean the
leak isn't real; it means this toy example is a best-case scenario for how little it could matter. The
leak stops being invisible the moment scale actually affects the result -- which is exactly the case with
Ridge/Lasso (Part 5), where the penalty is applied to the coefficients directly, so what scale they're on
matters a great deal; it's also the case with one-hot encoding of rare categories or imputed missing
values, where a fold's held-out rows can shift which categories or fill values the "leaked" version sees
as common.

The rule going forward, regardless of whether this particular pair of numbers matched: any preprocessing
that gets *fit* to data belongs inside a `Pipeline`, before you ever call `cross_val_score` or
`GridSearchCV` on it -- not applied by hand beforehand.

---

## Part 4 — Hyperparameter Tuning with GridSearchCV

So far, choices like "which `k` for k-NN" or "which `max_depth` for a tree" have been picked by us,
manually, more or less by guessing. `GridSearchCV` automates that search: give it a `Pipeline` and a grid
of hyperparameter values to try, and it fits and cross-validates a model for *every* combination, then
reports which one scored best -- all using only the training data.

The discipline stays the same as always: split into train/test **once**. Hand `GridSearchCV` the training
data only; it runs its own internal cross-validation folds within that. Look at the test set exactly once,
at the very end, after tuning is completely finished.

We'll tune a k-NN classifier on real NYC 311 data -- the same "will this complaint be late?" problem from
Lecture 9.

In [ ]:
import os

try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(8000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng2 = np.random.default_rng(383)
    n_rows = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_idx = rng2.integers(0, n_days, size=n_rows)
    created = days[day_idx] + pd.to_timedelta(rng2.integers(0, 24 * 60, size=n_rows), unit="m")
    still_open = rng2.random(n_rows) < 0.05
    resolution_hours = rng2.gamma(shape=1.5, scale=22.0, size=n_rows)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng2.choice(complaint_types, size=n_rows),
        "borough": rng2.choice(boroughs_list, size=n_rows, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour

complaints_df = complaints_df.dropna(subset=["resolution_time_hours"]).reset_index(drop=True)
standard_boroughs = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
complaints_df = complaints_df[complaints_df["borough"].isin(standard_boroughs)].reset_index(drop=True)
top_types = complaints_df["complaint_type"].value_counts().nlargest(15).index
complaints_df["complaint_grouped"] = complaints_df["complaint_type"].where(
    complaints_df["complaint_type"].isin(top_types), "Other"
)

# Same target as Lecture 9: did this complaint take more than 3 days (72 hours) to close?
complaints_df["is_late"] = (complaints_df["resolution_time_hours"] > 72).astype(int)

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
print(f"'Late' (>72h) rate: {complaints_df['is_late'].mean():.1%}")

In [ ]:
X = complaints_df[["hour_filed", "borough", "complaint_grouped"]]
y = complaints_df["is_late"]

X_train, X_test, y_train, y_test = __________(X, y, test_size=0.2, random_state=383, stratify=y)

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), ["hour_filed"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["borough", "complaint_grouped"]),
])

print("Training rows:", len(X_train), "| Test rows:", len(X_test))

In [ ]:
knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier()),
])

param_grid = {"classifier__n_neighbors": [3, 5, 9, 15, 25, 41]}

# Lecture 9 taught us accuracy isn't always the right thing to optimize on imbalanced data -- "late" is
# the rare, costly class here, so we ask GridSearchCV to pick the n_neighbors that maximizes RECALL on
# the positive class, not accuracy.
grid_search = __________(knn_pipeline, param_grid, cv=5, scoring="recall")
grid_search.fit(X_train, y_train)

print("Best n_neighbors:", grid_search.best_params_["classifier__n_neighbors"])
print(f"Best CV recall: {grid_search.best_score_:.3f}")

`GridSearchCV` tried all six values of `n_neighbors`, cross-validated each one (5 folds apiece -- 30 model
fits, from one `.fit()` call), and reported the value that scored best on average. Note the parameter name:
`"classifier__n_neighbors"` -- two underscores, `stepname__paramname`, pointing at the `n_neighbors`
argument of the step named `"classifier"` inside the pipeline. That's how you tell `GridSearchCV` which
knob, on which step of the pipeline, to turn.

`grid_search` has already refit the best pipeline on the *entire* training set behind the scenes -- it's
ready to predict on new data right now, no extra step needed.

In [ ]:
y_pred = grid_search.__________(X_test)
print(classification_report(y_test, y_pred, target_names=["on time", "late"]))

That's the one and only time the test set was touched in this whole section -- after every hyperparameter
decision was already locked in using cross-validation on the training data alone. If the test set had
been used to help *pick* `n_neighbors`, the resulting score would no longer be an honest estimate of how
the model performs on data it's never seen.

---

## Part 5 — Ridge and Lasso: Regularizing a Linear Model

A plain linear regression has no guardrails: given enough features (especially noisy or correlated ones),
it will happily assign a large coefficient to a feature that has no real relationship to the target, just
because doing so reduces error on the *training* data it can see. That's overfitting again, this time
showing up as unstable, inflated coefficients rather than a memorized decision boundary.

**Regularization** fights this by adding a penalty for large coefficients directly into what the model is
trying to minimize:

- **Ridge** (L2) penalizes the sum of *squared* coefficients. It shrinks every coefficient toward zero,
  proportionally, but rarely all the way to exactly zero.
- **Lasso** (L1) penalizes the sum of *absolute* coefficients. It can shrink some coefficients all the way
  to exactly zero -- effectively dropping those features from the model. That makes Lasso useful for
  automatic feature selection, not just shrinkage.

Both have a hyperparameter, `alpha`, that controls how strong the penalty is: `alpha=0` behaves like plain
linear regression; a very large `alpha` shrinks everything toward zero, including the real signal, which
underfits.

In [ ]:
rng3 = np.random.default_rng(383)
n_reg = 60

# One feature with a real, strong relationship to y ...
x_signal = rng3.normal(0, 1, n_reg)
# ... and eight features that are PURE noise, with no real relationship to y at all.
noise_features = rng3.normal(0, 1, size=(n_reg, 8))

y_reg = 4 * x_signal + rng3.normal(0, 1, n_reg)

reg_X = np.column_stack([x_signal, noise_features])
reg_feature_names = ["signal"] + [f"noise_{i+1}" for i in range(8)]

print(f"{n_reg} rows, 9 features (1 real signal + 8 pure noise)")

In [ ]:
plain_model = LinearRegression()
plain_model.__________(reg_X, y_reg)

for name, coef in zip(reg_feature_names, plain_model.coef_):
    print(f"{name:>10s}: {coef:+.3f}")

The `signal` coefficient should land somewhere near its true value (4.0). But look at the `noise_*`
coefficients -- they have no real relationship to `y_reg` at all, and yet plain linear regression still
assigns several of them a non-trivial, non-zero value. With only 60 rows and 9 features, the model has
just enough freedom to "explain" some of the noise by leaning on features that shouldn't matter.

In [ ]:
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(reg_X, y_reg)

lasso_model = __________(alpha=1.0)
lasso_model.fit(reg_X, y_reg)

coef_compare = pd.DataFrame({
    "feature": reg_feature_names,
    "plain": plain_model.coef_,
    "ridge (alpha=1.0)": ridge_model.coef_,
    "lasso (alpha=1.0)": lasso_model.coef_,
}).round(3)
coef_compare

Same data, three models. Ridge shrinks *every* coefficient toward zero a little, including `signal`'s --
but `signal` barely moves, because Ridge's penalty on squared coefficients doesn't mind one large
coefficient much. Lasso goes further: at this `alpha`, *every single* `noise_*` coefficient lands at
exactly 0.000 -- Lasso decided none of them contribute anything and dropped them completely. `signal`'s
coefficient shrinks noticeably more under Lasso than under Ridge (Lasso's penalty bites into large
coefficients too, not just small ones), but it's still, by a wide margin, the largest number in the
column -- Lasso traded a bit of accuracy on the one real feature for zeroing out eight fake ones.

That's the practical difference to remember: Ridge shrinks everything a little and rarely reaches zero;
Lasso can zero features out completely, at some cost to the features it keeps.

In [ ]:
alphas = np.logspace(-2, 3, 30)
ridge_coefs = []
for a in alphas:
    ridge_a = __________(alpha=a)
    ridge_a.fit(reg_X, y_reg)
    ridge_coefs.append(ridge_a.coef_)
ridge_coefs = np.array(ridge_coefs)

plt.figure(figsize=(7, 5))
for i, name in enumerate(reg_feature_names):
    is_signal = (name == "signal")
    plt.plot(alphas, ridge_coefs[:, i],
              label="signal" if is_signal else ("noise features" if i == 1 else None),
              color="#2B6CB0" if is_signal else "#9AA5B1",
              linewidth=2.5 if is_signal else 1)
plt.xscale("log")
plt.xlabel("alpha (regularization strength, log scale)")
plt.ylabel("coefficient value")
plt.title("Ridge: Coefficients Shrink as alpha Grows")
plt.legend()
plt.show()

Reading left to right (small `alpha` to large): every coefficient shrinks toward zero, but the `signal`
coefficient (blue) starts far larger than the noise coefficients (gray) and stays meaningfully above them
for most of the range -- it takes a much larger `alpha` to flatten it. Push `alpha` far enough, though,
and even `signal` gets crushed toward zero: too much regularization underfits the real relationship, not
just the noise.

So `alpha` itself is a hyperparameter that needs tuning, not a fixed setting -- which is exactly what
`GridSearchCV` from Part 4 is for. Part 6 does exactly that, on real data.

---

## Part 6 — Putting It Together: A Tuned Pipeline on Real NYC 311 Data

Same NYC 311 data as Part 4, same `complaints_df` -- but back to Lecture 7's original regression target,
`resolution_time_hours`. This time: `Pipeline` + `GridSearchCV` over `alpha`, and an honest comparison
against Lecture 7's plain linear regression baseline.

In [ ]:
X_reg_311 = complaints_df[["hour_filed", "borough", "complaint_grouped"]]
y_reg_311 = complaints_df["resolution_time_hours"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = __________(
    X_reg_311, y_reg_311, test_size=0.2, random_state=383
)

print("Training rows:", len(X_train_reg), "| Test rows:", len(X_test_reg))

In [ ]:
baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()),
])
baseline_pipeline.__________(X_train_reg, y_train_reg)
baseline_pred = baseline_pipeline.predict(X_test_reg)

baseline_rmse = mean_squared_error(y_test_reg, baseline_pred) ** 0.5
baseline_mae = mean_absolute_error(y_test_reg, baseline_pred)
baseline_r2 = r2_score(y_test_reg, baseline_pred)

print(f"Baseline (plain LinearRegression) -- RMSE: {baseline_rmse:.2f}  MAE: {baseline_mae:.2f}  R²: {baseline_r2:.3f}")

In [ ]:
ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge()),
])

param_grid_ridge = {"regressor__alpha": [0.1, 1, 10, 50, 100, 300]}

ridge_grid = __________(ridge_pipeline, param_grid_ridge, cv=5, scoring="r2")
ridge_grid.fit(X_train_reg, y_train_reg)

print("Best alpha:", ridge_grid.best_params_["regressor__alpha"])
print(f"Best CV R²: {ridge_grid.best_score_:.3f}")

In [ ]:
ridge_pred = ridge_grid.__________(X_test_reg)

ridge_rmse = mean_squared_error(y_test_reg, ridge_pred) ** 0.5
ridge_mae = mean_absolute_error(y_test_reg, ridge_pred)
ridge_r2 = r2_score(y_test_reg, ridge_pred)

print(f"Tuned Ridge                       -- RMSE: {ridge_rmse:.2f}  MAE: {ridge_mae:.2f}  R²: {ridge_r2:.3f}")
print(f"Baseline (plain LinearRegression) -- RMSE: {baseline_rmse:.2f}  MAE: {baseline_mae:.2f}  R²: {baseline_r2:.3f}")

With only three features here, plain linear regression was never at serious risk of overfitting, so
don't expect regularization to work miracles on this particular dataset -- the tuned Ridge model may only
match or slightly edge out the baseline. That's a fair result, not a failed one: the value of this whole
workflow (`Pipeline` + `GridSearchCV` + regularization) isn't a guaranteed score jump on three features,
it's that the exact same recipe scales unchanged to a problem with dozens or hundreds of features, where
overfitting risk is real and this discipline is what catches it.

---

### Putting the pieces together

```
DATA               split into train / test, ONCE
  |
  TRAIN  --> Pipeline (preprocessing + model)
                |
                cross_val_score / GridSearchCV   <- k-fold CV happens ONLY inside train
                |      (honest hyperparameter choice; test set never touched here)
                refit best pipeline on all of train
  |
  TEST   --> one honest evaluation, exactly once
```

`alpha` in Ridge/Lasso is just one more hyperparameter for `GridSearchCV` to search over -- the same
recipe as tuning `k` in k-NN (Part 4) or `max_depth` in a decision tree (Lecture 10). The model, the
tuning process, and the final evaluation are three different steps; keeping them in that order is what
makes the final number on the test set trustworthy.

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook:
`lect11_pipelines_tuning_exercise.ipynb`.

---

## Cheat Sheet

| Task | Code |
|---|---|
| Cross-validate a model | `cross_val_score(model, X, y, cv=5, scoring="r2")` |
| Bundle preprocessing + model | `Pipeline([("preprocessor", ct), ("model", est)])` |
| Search hyperparameters | `GridSearchCV(pipeline, param_grid, cv=5, scoring=...)` |
| Fit the search | `grid_search.fit(X_train, y_train)` |
| Best hyperparameters found | `grid_search.best_params_` |
| Best cross-validated score | `grid_search.best_score_` |
| Predict with the best model | `grid_search.predict(X_test)` |
| Pipeline param name in a grid | `"stepname__paramname"` |
| Ridge regression | `Ridge(alpha=...)` |
| Lasso regression | `Lasso(alpha=...)` |

---

## Key Terms

- **Cross-validation (k-fold)**: splitting data into *k* folds and rotating which one is held out for
  testing, then averaging the *k* scores -- a more stable performance estimate than one train/test split.
- **Pipeline**: bundles preprocessing steps and a model into one object with a single `.fit()` /
  `.predict()`; when cross-validated or tuned, its preprocessing is refit fresh on each fold's training
  portion automatically.
- **Data leakage (cross-validation version)**: fitting a preprocessing step on data outside the current
  fold's training portion, letting information about the held-out portion quietly influence the score.
- **GridSearchCV**: tries every combination of hyperparameter values in a grid, scores each with
  cross-validation, and reports the best -- all without touching the test set.
- **Hyperparameter**: a setting chosen before training (like `k` in k-NN or `alpha` in Ridge) rather than
  learned from data during training.
- **Regularization**: adding a penalty for large coefficients to what a linear model minimizes, to
  discourage overfitting.
- **Ridge (L2 regularization)**: penalizes the sum of squared coefficients; shrinks every coefficient
  toward zero, rarely exactly to zero.
- **Lasso (L1 regularization)**: penalizes the sum of absolute coefficients; can shrink some coefficients
  to exactly zero, acting as automatic feature selection.
- **alpha**: the hyperparameter controlling regularization strength in Ridge/Lasso; `alpha=0` behaves like
  plain linear regression, larger `alpha` shrinks coefficients more aggressively.